In [1]:
import numpy as np
import pandas as pd
from configuration import Config

# Utility Functions

In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    """Loads a dataset from a CSV file."""
    print(f"Loading dataset: {filename}")
    return pd.read_csv(filename)

In [3]:
def save_dataset(df: pd.DataFrame, filename: str):
    """Saves a DataFrame to a CSV file."""
    print(f"Saving dataset to: {filename}")
    df.to_csv(filename, index=False)

In [4]:
def print_dataframe_stats(df: pd.DataFrame, df_name: str = "DataFrame"):
    """Prints basic statistics about a DataFrame."""
    print(f"\n--- Stats for {df_name} ---")
    print(f"Shape: {df.shape}")
    print(f"Number of rows with missing values: {df.isnull().any(axis=1).sum()}")
    print(f"Number of columns with missing values: {df.isnull().any(axis=0).sum()}")
    print("---------------------------\n")

# Feature engineering functions

In [ ]:
def add_engineered_features(df_raw: pd.DataFrame, price_cap: float) -> pd.DataFrame:
    """Adds new features to the DataFrame based on domain knowledge or data exploration."""
    df = df_raw.copy()

    # Cap price and handle potential division by zero or invalid values
    df.loc[df['price_usd'] > price_cap, 'price_usd'] = np.nan # Cap price
    df['price_usd'] = np.where(df['price_usd'] <= 0, np.nan, df['price_usd']) # Ensure price is positive
    
    # Ensure srch_length_of_stay is positive before division
    valid_stay = (df['srch_length_of_stay'].notna()) & (df['srch_length_of_stay'] > 0)
    df['price_per_night'] = np.nan
    df.loc[valid_stay, 'price_per_night'] = df.loc[valid_stay, 'price_usd'] / df.loc[valid_stay, 'srch_length_of_stay']
    df['month'] = pd.to_datetime(df['date_time'], errors='coerce').dt.month

    # Relative features (handle cases where median might be NaN or group is small)
    for group_key, feature, new_feature_name in [
        (Config.GROUP_COL, 'prop_review_score', 'review_score_relative'),
        (Config.GROUP_COL, 'price_usd', 'price_relative')
    ]:
        # Calculate median per group, ensuring it's a scalar or aligns correctly
        median_per_group = df.groupby(group_key)[feature].transform('median')
        df[new_feature_name] = df[feature] - median_per_group

    # Log-transformed features (add 1 to handle zeros, ensure positive values)
    df['log_price_usd'] = np.log1p(df['price_usd'].fillna(0).clip(lower=0))
    df['log_price_per_night'] = np.log1p(df['price_per_night'].fillna(0).clip(lower=0))

    # Missing value indicators
    df['orig_dest_missing'] = df['orig_destination_distance'].isnull().astype(int)
    df['loc_score2_missing'] = df['prop_location_score2'].isnull().astype(int)
    
    # Fill NaNs created by operations if necessary, or handle them downstream
    # For example, log features might produce -inf if original was 0 before log1p, or NaN if original was negative.
    # log1p handles 0 correctly (log(1)=0). If price_usd can be 0, log1p is fine.
    # If price_usd can be negative (which it shouldn't be), clip(lower=0) handles that.
    df = df.replace([np.inf, -np.inf], np.nan) # Replace infs that might arise
    return df

In [6]:
def compute_property_statistics(df: pd.DataFrame, group_by_col: str = Config.PROP_ID_COL) -> pd.DataFrame:
    """Computes aggregate statistics (mean, median) for properties."""
    # Define numeric columns for aggregation, excluding IDs, flags, and target-like features
    # This list should be curated based on the dataset.
    cols_to_exclude = [
        Config.GROUP_COL, Config.PROP_ID_COL, 'position', 'click_bool', 'booking_bool',
        'gross_bookings_usd', 'relevance', 'price_usd', 'visitor_location_country_id',
        'prop_country_id', 'site_id', 'srch_destination_id', 'date_time', # date_time is not numeric
        # Exclude engineered features that are relative or log-transformed if they don't make sense to average globally for a prop_id
        'price_per_night', 'month', 'review_score_relative', 'price_relative',
        'log_price_usd', 'log_price_per_night', 'orig_dest_missing', 'loc_score2_missing'
    ]
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    numeric_cols_to_agg = [col for col in numeric_cols if col not in cols_to_exclude]
    
    print(f"Aggregating stats for columns: {numeric_cols_to_agg}")
    means = df.groupby(group_by_col)[numeric_cols_to_agg].mean().add_suffix('_mean')
    medians = df.groupby(group_by_col)[numeric_cols_to_agg].median().add_suffix('_median')
    prop_stats = means.join(medians, how='outer') # Use outer join to keep all prop_ids
    return prop_stats.reset_index()

In [7]:
def merge_property_statistics(df: pd.DataFrame, prop_stats: pd.DataFrame, on_col: str = Config.PROP_ID_COL) -> pd.DataFrame:
    """Merges property statistics back into the main DataFrame."""
    return df.merge(prop_stats, on=on_col, how='left')

In [8]:
def define_relevance(df: pd.DataFrame) -> pd.DataFrame:
    """Defines the relevance score based on click and booking booleans."""
    df_copy = df.copy()
    df_copy[Config.TARGET_COL] = 0
    df_copy.loc[df_copy['click_bool'] == 1, Config.TARGET_COL] = 1
    df_copy.loc[df_copy['booking_bool'] == 1, Config.TARGET_COL] = 5 # Higher relevance for booking
    return df_copy

In [9]:
def select_and_clean_features(df: pd.DataFrame, cols_to_drop_final: list, target_col: str, group_col: str) -> pd.DataFrame:
    """Selects final features and drops specified columns. Fills remaining NaNs with a placeholder (e.g., 0 or median)."""
    df_copy = df.copy()
    
    # Drop specified non-feature columns (original script dropped these before saving _stats files)
    df_copy = df_copy.drop(columns=cols_to_drop_final, errors='ignore')

    # Identify feature columns (all except target and group_col if they exist)
    feature_cols = [col for col in df_copy.columns if col not in [target_col, group_col, Config.PROP_ID_COL]] # Keep PROP_ID_COL for now if needed for submission join

    # Fill NaNs in feature columns - simple strategy (0). More sophisticated imputation could be used.
    for col in feature_cols:
        if df_copy[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df_copy[col]):
                df_copy[col] = df_copy[col].fillna(0) # Or use median: df_copy[col].fillna(df_copy[col].median(), inplace=True)
            else: # For non-numeric, fill with a placeholder like 'missing' or mode
                df_copy[col] = df_copy[col].fillna('missing')
    
    # Ensure no inf values remain
    df_copy = df_copy.replace([np.inf, -np.inf], 0) # Or another appropriate fill value
    return df_copy

In [13]:
def preprocess_data(
    raw_train_path: str,
    raw_test_path: str,
    processed_train_path: str,
    processed_test_path: str,
    prop_stats_path: str,
    read_from_file: bool,
    config: Config
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Full data preprocessing pipeline:
    1. Load raw data.
    2. Engineer features.
    3. Define relevance for training data.
    4. Compute property statistics (from training data only).
    5. Merge statistics into training and test data.
    6. Select final features and clean (handle NaNs).
    7. Save processed datasets and property statistics.
    """
    print("Starting data preprocessing...")

    # --- Training Data ---
    train_raw_df = load_dataset(raw_train_path)
    print_dataframe_stats(train_raw_df, "Raw Training Data")

    if read_from_file:
        prop_stats_df = load_dataset(prop_stats_path)
    else:            
        train_fe_df = add_engineered_features(train_raw_df, config.PRICE_CAP)
        train_fe_df = define_relevance(train_fe_df) # Define target for training set
        # Compute property statistics ONLY from the training data
        prop_stats_df = compute_property_statistics(train_fe_df, group_by_col=config.PROP_ID_COL)
        save_dataset(prop_stats_df, prop_stats_path)
    
    print_dataframe_stats(prop_stats_df, "Property Statistics")

    if read_from_file:
        train_processed_df = load_dataset(processed_train_path)
    else:
        train_merged_df = merge_property_statistics(train_fe_df, prop_stats_df, on_col=config.PROP_ID_COL)
        train_processed_df = select_and_clean_features(
            train_merged_df,
            config.COLS_TO_DROP_PRE_TRAIN,
            config.TARGET_COL,
            config.GROUP_COL
        )
        save_dataset(train_processed_df, processed_train_path)
    
    print_dataframe_stats(train_processed_df, "Processed Training Data")

    # --- Test Data ---
    test_raw_df = load_dataset(raw_test_path)
    print_dataframe_stats(test_raw_df, "Raw Test Data")
    test_fe_df = add_engineered_features(test_raw_df, config.PRICE_CAP)
    # Note: Relevance is not defined for the raw test set as it's unknown.

    # Merge property statistics (learned from training) into the test data
    test_merged_df = merge_property_statistics(test_fe_df, prop_stats_df, on_col=config.PROP_ID_COL)
    test_processed_df = select_and_clean_features(
        test_merged_df,
        config.COLS_TO_DROP_PRE_TRAIN,
        config.TARGET_COL, # Target col won't exist in test_raw, select_and_clean_features handles this
        config.GROUP_COL
    )
    save_dataset(test_processed_df, processed_test_path)
    print_dataframe_stats(test_processed_df, "Processed Test Data")
    print("Data preprocessing complete.")
    return train_processed_df, test_processed_df, prop_stats_df 

In [14]:
config = Config()
Config.create_dirs() # Ensure directories exist

train_processed_df, test_processed_df, _ = preprocess_data(
    raw_train_path=config.RAW_TRAINING_FILE,
    raw_test_path=config.RAW_TEST_FILE,
    processed_train_path=config.PROCESSED_TRAINING_FILE,
    processed_test_path=config.PROCESSED_TEST_FILE,
    prop_stats_path=config.PROP_STATS_FILE,
    read_from_file=True,
    config=config
)

Starting data preprocessing...
Loading dataset: ./datasets\training_set_VU_DM.csv

--- Stats for Raw Training Data ---
Shape: (4470838, 54)
Number of rows with missing values: 4470838
Number of columns with missing values: 31
---------------------------

Loading dataset: ./datasets\processed_data\prop_stats.csv

--- Stats for Property Statistics ---
Shape: (127739, 85)
Number of rows with missing values: 127739
Number of columns with missing values: 60
---------------------------

Loading dataset: ./datasets\processed_data\train_processed.csv

--- Stats for Processed Training Data ---
Shape: (3722553, 141)
Number of rows with missing values: 0
Number of columns with missing values: 0
---------------------------

Loading dataset: ./datasets\test_set_VU_DM.csv

--- Stats for Raw Test Data ---
Shape: (4959183, 50)
Number of rows with missing values: 4959183
Number of columns with missing values: 30
---------------------------

Saving dataset to: ./datasets\processed_data\test_processed.cs